# Lowpass Filtering & Thresholding for Region Extraction
**Hubble Space Telescope — Hickson Compact Group**

This notebook reproduces the three-step pipeline from the lecture:
- **(a)** Original 2566×2758 Hubble image
- **(b)** Result of lowpass filtering with a Gaussian kernel
- **(c)** Result of thresholding the filtered image (intensities scaled to [0, 1])

## 1. Install & Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from skimage import io, img_as_float, filters
from skimage.color import rgb2gray
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## 2. Upload the Hubble Image
Click **Choose Files** and upload your Hubble Telescope image.

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
image_raw = io.imread(filename)

print(f'Loaded: {filename}')
print(f'Shape : {image_raw.shape}')
print(f'Dtype : {image_raw.dtype}')

## 3. Convert to Grayscale Float [0, 1]

In [ ]:
if image_raw.ndim == 3:
    image_gray = rgb2gray(img_as_float(image_raw))
else:
    image_gray = img_as_float(image_raw)

print(f'Grayscale shape : {image_gray.shape}')
print(f'Intensity range : [{image_gray.min():.4f}, {image_gray.max():.4f}]')

plt.figure(figsize=(7, 7))
plt.imshow(image_gray, cmap='gray')
plt.title(f'(a) Original Image — {image_gray.shape[1]}×{image_gray.shape[0]}')
plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Apply Gaussian Lowpass Filter
The Gaussian kernel suppresses high-frequency noise, leaving only large bright structures (galaxies, star clusters).

**Tune `sigma`** — larger value = more smoothing.

In [ ]:
sigma = 3   # <-- adjust this

image_filtered = gaussian_filter(image_gray, sigma=sigma)

print(f'Gaussian filter applied  sigma={sigma}')
print(f'Filtered range : [{image_filtered.min():.4f}, {image_filtered.max():.4f}]')

plt.figure(figsize=(7, 7))
plt.imshow(image_filtered, cmap='gray')
plt.title(f'(b) Gaussian Lowpass Filter  (sigma={sigma})')
plt.axis('off')
plt.tight_layout()
plt.show()

## 5. Threshold the Filtered Image (intensities scaled to [0, 1])
Pixels above `threshold_value` become white (extracted regions), rest become black.

**Tune `threshold_value`** — lower = more regions detected.

In [ ]:
# Scale filtered image to [0, 1]
image_normalized = (image_filtered - image_filtered.min()) / \
                   (image_filtered.max() - image_filtered.min())

threshold_value = 0.15   # <-- adjust this (0.0 – 1.0)

# Optional: Otsu automatic threshold (uncomment to use)
# threshold_value = filters.threshold_otsu(image_normalized)
# print(f'Otsu threshold: {threshold_value:.4f}')

image_thresholded = image_normalized > threshold_value

print(f'Threshold        : {threshold_value}')
print(f'Extracted pixels : {image_thresholded.sum()} ({100*image_thresholded.mean():.2f}%)')

plt.figure(figsize=(7, 7))
plt.imshow(image_thresholded, cmap='gray')
plt.title(f'(c) Thresholded Image  (threshold={threshold_value}, scaled [0,1])')
plt.axis('off')
plt.tight_layout()
plt.show()

## 6. Final Combined Result — (a), (b), (c)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Lowpass Filtering & Thresholding — Hickson Compact Group (Hubble Space Telescope)',
    fontsize=13, fontweight='bold'
)

axes[0].imshow(image_gray, cmap='gray')
axes[0].set_title(f'(a) Original\n{image_gray.shape[1]}×{image_gray.shape[0]} px', fontsize=11)
axes[0].axis('off')

axes[1].imshow(image_filtered, cmap='gray')
axes[1].set_title(f'(b) Gaussian Lowpass\nsigma={sigma}', fontsize=11)
axes[1].axis('off')

axes[2].imshow(image_thresholded, cmap='gray')
axes[2].set_title(f'(c) Thresholded\nthreshold={threshold_value}, scaled [0,1]', fontsize=11)
axes[2].axis('off')

plt.tight_layout()
plt.savefig('result_lowpass_thresholding.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: result_lowpass_thresholding.png')

## 7. (Bonus) Compare Different Sigma Values

In [ ]:
sigmas = [1, 3, 5, 10, 20]

fig, axes = plt.subplots(2, len(sigmas), figsize=(20, 8))
fig.suptitle('Effect of Different Gaussian Sigma Values', fontsize=14, fontweight='bold')

for i, s in enumerate(sigmas):
    f = gaussian_filter(image_gray, sigma=s)
    n = (f - f.min()) / (f.max() - f.min())
    binary = n > threshold_value

    axes[0, i].imshow(f, cmap='gray')
    axes[0, i].set_title(f'Filtered  σ={s}', fontsize=10)
    axes[0, i].axis('off')

    axes[1, i].imshow(binary, cmap='gray')
    axes[1, i].set_title(f'Threshold  σ={s}', fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('result_sigma_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. (Bonus) Compare Different Threshold Values

In [ ]:
thresholds = [0.05, 0.10, 0.15, 0.20, 0.30]
f = gaussian_filter(image_gray, sigma=sigma)
n = (f - f.min()) / (f.max() - f.min())

fig, axes = plt.subplots(1, len(thresholds), figsize=(20, 4))
fig.suptitle(f'Effect of Different Threshold Values  (sigma={sigma})', fontsize=14)

for i, t in enumerate(thresholds):
    binary = n > t
    axes[i].imshow(binary, cmap='gray')
    axes[i].set_title(f't={t}\n{binary.sum()} px', fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('result_threshold_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Download Results

In [ ]:
files.download('result_lowpass_thresholding.png')
files.download('result_sigma_comparison.png')
files.download('result_threshold_comparison.png')
print('Done! All results downloaded.')